<a href="https://colab.research.google.com/github/isaacbentley/sigint-rf-skill/blob/main/examples/run_agent_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📻 SigInt RF Agent: Demodulation Without the DSP Drudgery

Welcome to the interactive demo for the **SigInt RF Agentic Skill**, presented at DefCon!

This notebook runs entirely in your browser. You don't need to install GNU Radio, you don't need to fight with dependencies, and you don't need a math degree.

### What does this do?
1. **Ingest**: Upload a raw IQ capture from your SDR (or use our provided demo capture).
2. **Triage**: Our local DSP script extracts the physical layer metadata (Bandwidth, SNR, Autocorrelation, Phase Clusters).
3. **Exploit**: We feed those parameters into an AI Agent loaded with our custom RF security persona. The agent will identify the protocol, flag security vulnerabilities (like missing rolling codes), and generate a custom Python script to extract the binary payload.

> [!TIP]
> **DefCon Challenge**: We've included a `mystery_capture.cf32` file in the repo. Run it through this notebook. If you can decode the flag, come find me after the talk!

Ready? Click the **Play** buttons on the hidden code cells below to get started.

In [ ]:
# @title 1. Setup Environment { display-mode: "form" }
!pip install -q google-genai numpy scipy matplotlib ipywidgets
import os
if not os.path.exists('sigint-rf-skill'):
    !git clone https://github.com/isaacbentley/sigint-rf-skill.git

output_dir = 'sigint-rf-skill/captures/'
os.makedirs(output_dir, exist_ok=True)

print("Generating demo files...")
!python3 sigint-rf-skill/tools/generate_demo_signal.py --type gmsk --duration 0.5 --sample_rate 2048000 --output_file {output_dir}mystery_capture.cf32 > /dev/null
print("✅ Environment setup and demo files ready.")

In [ ]:
# @title 2. Select Capture File { display-mode: "form" }
target_file = 'sigint-rf-skill/captures/mystery_capture.cf32' #@param ['sigint-rf-skill/captures/mystery_capture.cf32']
print(f"Selected file for triage: {target_file}")

In [ ]:
# @title 3. Run Triage App and Open Report { display-mode: "form" }
import os
from IPython.display import Markdown, Image, display

if not os.path.exists(target_file):
    print(f"❌ Error: {target_file} not found. Please upload a valid capture.")
else:
    print(f"Running local triage tool on {target_file}...")
    !python3 sigint-rf-skill/tools/triage_iq.py --file {target_file} --rate 2048000 > /dev/null
    
    print("\n📊 **Local DSP Triage Results:**")
    if os.path.exists('triage_plot.png'):
        display(Image('triage_plot.png'))
    
    try:
        with open('triage_report.md', 'r') as f:
            triage_report = f.read()
        display(Markdown(triage_report))
    except FileNotFoundError:
        print("❌ Error: No triage report found.")

In [ ]:
# @title 4. AI Demodulation Report { display-mode: "form" }
from google.colab import ai
from IPython.display import Markdown, display

with open('sigint-rf-skill/SKILL.md', 'r') as f:
    system_instruction = f.read()

try:
    with open('triage_report.md', 'r') as f:
        triage_report = f.read()
except FileNotFoundError:
    triage_report = "[Error: No triage report found.]"
    
prompt = f"""System Instructions:
{system_instruction}

---------------------
User Request:
I have run the triage tool on my signal. Here is the output report:

{triage_report}

Based on this report, please analyze the signal, provide a demodulation snippet, and highlight any security vulnerabilities."""

print("🧠 Sending report to Google Colab AI...")
try:
    # Attempt to use the requested gemini-2.5-flash model
    response = ai.generate_text(prompt, model_name='google/gemini-2.5-flash')
except Exception as e:
    print(f"⚠️ Could not use gemini-2.5-flash ({e}). Falling back to default model...")
    response = ai.generate_text(prompt)

display(Markdown(str(response)))


In [ ]:
# @title 5. Execute Explainable Demodulator { display-mode: "form" }
import os
from IPython.display import Image, display

print(f"Extracting binary payload from {target_file}...")
print("-" * 50)
!python3 sigint-rf-skill/tools/explainable_demod.py --file {target_file} --rate 2048000 --mode fsk --symbol-rate 100000
print("-" * 50)
print("✅ Payload extraction complete!")

if os.path.exists('demod_diagnostics.png'):
    print("\n🎨 **DSP Hybrid Demodulation Diagnostics:**")
    display(Image('demod_diagnostics.png'))